# Predicción de precios de productos

In [ ]:
!pip install -q datasets==2.21.0
!pip install -q transformers==4.43.1 trl==0.9.6 peft==0.12.0 accelerate==0.32.1
!pip install -q --upgrade bitsandbytes
!pip install -q triton==3.1.0
!pip install numpy==1.26.4 --force-reinstall


In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Datos del proyecto, data y usuario de HF
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer"
HF_USER = "PamAmezcua"
DATASET_NAME = f"{HF_USER}/lite-data"
MAX_SEQUENCE_LENGTH = 182
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}" #fecha actual
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}" #nombre del proyecto es pricer-fecha
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hiper parámetros de QLoRA
LORA_R = 32 #con gpu pequeña: 8. Rango bajo de la matriz
LORA_ALPHA = 64 #con gpu pequeña: 16.
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"] #módulos a los que apuntamos con las matrices de loora y para los que los pesos se van a ajustar, en este caso las capas de atención del transformer.
LORA_DROPOUT = 0.1 #tomamos el 10% de las neuronas de forma aleatoria para que no trabajen.
QUANT_4_BIT = True

# Hiper parámetros para el Entrenamiento
EPOCHS = 3
BATCH_SIZE = 8 #2, 4, 16, tambien cambia segun el tamaño de gpu, a menor gpu menor lote. Lotes de ejemplos que el modelo procesa al mismo tiempo antes de actualizar matrices loora de pesos. un step es la actualizacion de pesos usando batch_size ejemplos
GRADIENT_ACCUMULATION_STEPS = 2 #2, 4. Acumula gradientes de 2 batches (16 ejemplos) antes de actualizar, simulando un batch más grande sin usar más memoria.
LEARNING_RATE = 1e-4 #tasa del velocidad ("tamaño" del paso) hacia el punto de minimizacion del error, con el metodo de gradiente. Si es un paso muy grande puede que nos saltemos el valle, por eso es tan pequeño.
                    #el paso puede ser que sea tan pequeño que no logre salir de cierto valle y que sea mínimo local no global (problemático dejar de ver el minimo global por estar atorado en el minimo local).
                    #podemos usar un "programador del learning rate" segun la epoc en la que estamos.
LR_SCHEDULER_TYPE = 'cosine' #la funcion coseno es bajo la cual vamos a ir bajando el valor del ratio de aprendizaje.
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Configuración de Admin
STEPS = 50 #Cada 50 steps va generando la pérdida del modelo
SAVE_STEPS = 500 #Cada cuántos steps se guardan las matrices loora. Cada step a su vez tiene 16 elementos
LOG_TO_WANDB = True #registro de weigths and biases.

%matplotlib inline

# Siempre se hace shuffle antes de entrenar pues El shuffle garantiza que cada batch sea una muestra representativa de todo el dataset.
#dataset = dataset.shuffle(seed=42) #seed=42 determina en qué orden se ven los ejemplos.

In [ ]:
HUB_MODEL_NAME

In [ ]:
# Log in en HuggingFace
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in en Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configuramos Weights & Biases para almacenar la info de nuestro proyecto
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

In [ ]:
DATASET_NAME

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

len(train), len(test)

In [ ]:
test[0]

In [ ]:
#Para arrancar proyecto nuevo en WB
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# Modelo base

In [ ]:
#Cuantización
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [ ]:
# Cargamos el Tokenizer y el Modelo

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
#Enmascaramiento sencillo del modelo, para acotar cómo debe dar las respuestas
from trl import DataCollatorForCompletionOnlyLM #DataCollatorForCompletionOnlyLM: esta clase nos brinda una forma de hacer enmascaramientos sencillos
response_template = "Price is $" #frase que necesito que el modelo complete, se le acota el entrenamiento que va a realizar
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# Configuración para el entrenamiento

Necesitamos crear 2 objetos:

1) Un objeto LoraConfig con nuestros hiperparámetros para LoRA

2) Un SFTConfig con nuestros parámetros generales de entrenamiento

In [ ]:
# Primero, especificar los parámetros de configuración para LoRA

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

#A continuación, especificar los parámetros de configuración general para el entrenamiento.
# Supervised Fine-Tuning (Ajuste Fino Supervisado)
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, #recordemos: tamaño del lote
    per_device_eval_batch_size=1,
    eval_strategy="no", #sin conjunto de evaluacion, podriamos pasarle el conjunto de test.
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True, #sube a HF cada save_steps
    hub_model_id=HUB_MODEL_NAME, #que se guarde en HF
    hub_private_repo=True #modelo privado
)

# Y ahora, el entrenador de ajuste fino supervisado realizará el ajuste fino
# Dados estos 2 conjuntos de parámetros de configuración

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    tokenizer=tokenizer,
    args=train_parameters,
    data_collator=collator #mascara
)

# Entrenamiento

In [ ]:
fine_tuning.train()

# Subimos nuestro modelo ajustado a Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Almacenado en el hub: {PROJECT_RUN_NAME}")

In [ ]:
# Tabla del detalle por epoch

history = fine_tuning.state.log_history

# verlo como tabla
import pandas as pd
df = pd.DataFrame(history)
print(df)

# guardarlo como CSV
df.to_csv("training_loss.csv", index=False)

In [ ]:
x=df["step"]
y=df["loss"]

plt.plot(x, y)
plt.axvline(x=1550, color='red', linestyle='--', linewidth=1.5) #epoch 0.992
plt.axvline(x=3100, color='red', linestyle='--', linewidth=1.5) #epoch 1984

plt.title('Step vs Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
# Mostrar la gráfica
plt.show()

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()